# CSE 164 A100 Supervised Multi-Task Runs

This notebook is for Google Colab A100 runs. It keeps the same Drive data layout as the existing Colab notebook:

```text
/content/drive/MyDrive/CSE164FinalProject/raw/
|-- metadata/
|-- test/
|-- train_labeled/
|-- train_seg/
|-- train_unlabeled/
`-- val/
```

Outputs are written to `/content/drive/MyDrive/CSE164FinalProject/outputs/` so checkpoints and submissions survive runtime resets.

In [33]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!nvidia-smi

Thu Jun 11 20:04:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   24C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Dataset set up

In [3]:
!mkdir -p /content/data

!cp /content/drive/MyDrive/CSE164FinalProject/raw.zip /content/raw.zip

!unzip -q /content/raw.zip -d /content/data

!ls /content/data
!ls /content/data/raw

raw
metadata  test	train_labeled  train_seg  train_unlabeled  val


# Clone or update Repo

In [53]:
import getpass

token = getpass.getpass("GitHub Token: ")

if REPO_DIR.exists():
    %cd {REPO_DIR}
    !git pull
else:
    %cd /content
    !git clone https://{token}@github.com/007deepaks/CSE-164FinalProject.git
    %cd {REPO_DIR}

!git status --short

GitHub Token: ··········
/content/CSE-164FinalProject
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 960 bytes | 960.00 KiB/s, done.
From https://github.com/007deepaks/CSE-164FinalProject
   5b4d892..7e89dd8  main       -> origin/main
Updating 5b4d892..7e89dd8
Fast-forward
 src/training/train_multitask_fixmatch.py | 32 +++++++++++++++++++++++++++++++-
 1 file changed, 31 insertions(+), 1 deletion(-)


## Repository and Paths

If the repo is not already in `/content/CSE-164FinalProject`, upload it, clone it, or copy it from Drive before continuing.

In [34]:
from pathlib import Path
import os

REPO_DIR = Path('/content/CSE-164FinalProject')

# local SSD (fast)
DATA_ROOT = Path('/content/data/raw')

# checkpoints still saved to Drive
DRIVE_PROJECT = Path('/content/drive/MyDrive/CSE164FinalProject')
DRIVE_OUTPUTS = DRIVE_PROJECT / 'outputs'

assert REPO_DIR.exists(), f'Missing repo at {REPO_DIR}'
assert DATA_ROOT.exists(), f'Missing data at {DATA_ROOT}'

DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)

os.chdir(REPO_DIR)

print('repo:', REPO_DIR)
print('data:', DATA_ROOT)
print('drive outputs:', DRIVE_OUTPUTS)

repo: /content/CSE-164FinalProject
data: /content/data/raw
drive outputs: /content/drive/MyDrive/CSE164FinalProject/outputs


In [ ]:
!python -m pip install -q -r requirements.txt
!python -m pip install -q kaggle

In [8]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
import torch
print(torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
2.11.0+cu128
cuda available: True
NVIDIA RTX PRO 6000 Blackwell Server Edition


## Serious ResNet-50 Supervised Run

This trains ResNet-50 from scratch with shared encoder, U-Net binary decoder, mask-guided classifier pooling, EMA, warmup plus cosine LR, and weighted supervised sampling.

In [15]:
!python -m src.training.train_multitask \
  --data-root "$DATA_ROOT" \
  --architecture resnet50 \
  --stage joint \
  --epochs 100 \
  --warmup-epochs 3 \
  --image-size 384 \
  --num-segmentation-classes 1 \
  --seg-batch-size 6 \
  --cls-batch-size 32 \
  --val-batch-size 6 \
  --num-workers 4 \
  --learning-rate 1e-3 \
  --min-learning-rate 1e-6 \
  --weight-decay 5e-2 \
  --label-smoothing 0.1 \
  --segmentation-loss-weight 1.0 \
  --dice-loss-weight 1.0 \
  --seg-classification-loss-weight 1.0 \
  --cls-loss-weight 1.0 \
  --gradient-clip 2.0 \
  --ema-decay 0.9998 \
  --weighted-combined-sampling \
  --class-only-sample-weight 1.0 \
  --mask-sample-weight 2.5 \
  --validation-threshold 0.50 \
  --checkpoint-dir "$DRIVE_OUTPUTS/checkpoints/resnet50_multitask_384"

Using device: cuda; mixed precision: True
Multi-task ResNet-50 from scratch: image_size=384, decoder=unet, num_segmentation_classes=1, classifier_dropout=0.2
Train samples: segmentation=3000, classification=10500; val=750
LR scheduler: linear warmup for 3 epochs to 0.001, then cosine decay to 1e-06
Training stage: joint

Epoch 1/100
  mixed batch 0020/500 seg_loss=7.5150 cls_loss=6.2072
  mixed batch 0040/500 seg_loss=7.9921 cls_loss=6.4041
  mixed batch 0060/500 seg_loss=7.7368 cls_loss=6.1519
  mixed batch 0080/500 seg_loss=7.2428 cls_loss=6.6187
  mixed batch 0100/500 seg_loss=6.7928 cls_loss=6.1494
  mixed batch 0120/500 seg_loss=7.5328 cls_loss=6.3534
  mixed batch 0140/500 seg_loss=7.5647 cls_loss=6.3118
  mixed batch 0160/500 seg_loss=7.0334 cls_loss=6.8707
  mixed batch 0180/500 seg_loss=6.7786 cls_loss=6.1946
  mixed batch 0200/500 seg_loss=8.2578 cls_loss=6.2588
  mixed batch 0220/500 seg_loss=8.1721 cls_loss=6.7904
  mixed batch 0240/500 seg_loss=8.1407 cls_loss=6.4477
  mix

## Threshold Sweep

Run after training finishes. Pick the threshold with the best automated validation score.

In [12]:
!python -m src.training.tune_multitask_threshold \
  --seg-checkpoint "$DRIVE_OUTPUTS/checkpoints/resnext50_32x4d_offline_pseudo_highconf/best_multitask.pt" \
  --data-root "$DATA_ROOT" \
  --image-size 448 \
  --batch-size 24 \
  --num-workers 12 \
  --tta hflip \
  --thresholds 0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70

  scored batch 0020/32
  scored batch 0032/32
threshold=0.300 automated=0.2270 seg=0.2133 mIoU=0.1956 boundary=0.3160 rare=0.1317 macro_acc=0.3883
threshold=0.350 automated=0.2278 seg=0.2145 mIoU=0.1963 boundary=0.3191 rare=0.1329 macro_acc=0.3883
threshold=0.400 automated=0.2284 seg=0.2153 mIoU=0.1967 boundary=0.3212 rare=0.1338 macro_acc=0.3883
threshold=0.450 automated=0.2288 seg=0.2159 mIoU=0.1971 boundary=0.3224 rare=0.1347 macro_acc=0.3883
threshold=0.500 automated=0.2290 seg=0.2162 mIoU=0.1973 boundary=0.3229 rare=0.1354 macro_acc=0.3883
threshold=0.550 automated=0.2291 seg=0.2163 mIoU=0.1974 boundary=0.3226 rare=0.1360 macro_acc=0.3883
threshold=0.600 automated=0.2290 seg=0.2162 mIoU=0.1975 boundary=0.3214 rare=0.1365 macro_acc=0.3883
threshold=0.650 automated=0.2288 seg=0.2158 mIoU=0.1975 boundary=0.3196 rare=0.1370 macro_acc=0.3883
threshold=0.700 automated=0.2283 seg=0.2151 mIoU=0.1973 boundary=0.3165 rare=0.1372 macro_acc=0.3883
BEST threshold=0.550 automated_score=0.2291


## Generate Submission

Replace `BEST_THRESHOLD` with the best threshold from the sweep.

In [17]:
BEST_THRESHOLD = 0.55

In [22]:
!python -m src.training.predict_multitask \
  --checkpoint "$DRIVE_OUTPUTS/checkpoints/resnext50_cls_finetune_fgcutmix_448/best_multitask.pt" \
  --data-root "$DATA_ROOT" \
  --image-size 448 \
  --batch-size 6 \
  --num-workers 4 \
  --tta hflip \
  --seg-threshold 0.55  \
  --output "$DRIVE_OUTPUTS/submissions/resnext50_cls_finetune_fgcutmix_448.csv"

  predicted batch 0020/500
  predicted batch 0040/500
  predicted batch 0060/500
  predicted batch 0080/500
  predicted batch 0100/500
  predicted batch 0120/500
  predicted batch 0140/500
  predicted batch 0160/500
  predicted batch 0180/500
  predicted batch 0200/500
  predicted batch 0220/500
  predicted batch 0240/500
  predicted batch 0260/500
  predicted batch 0280/500
  predicted batch 0300/500
  predicted batch 0320/500
  predicted batch 0340/500
  predicted batch 0360/500
  predicted batch 0380/500
  predicted batch 0400/500
  predicted batch 0420/500
  predicted batch 0440/500
  predicted batch 0460/500
  predicted batch 0480/500
  predicted batch 0500/500
Wrote /content/drive/MyDrive/CSE164FinalProject/outputs/submissions/resnext50_cls_finetune_fgcutmix_448.csv
{
  "rows": 3000,
  "scored": false,
  "status": "format ok"
}


In [19]:
!python starter/validate_submission_csv.py \
  --submission "$DRIVE_OUTPUTS/submissions/resnet50_multitask_384_tta.csv" \
  --data-root "$DATA_ROOT" \
  --split test

{
  "rows": 3000,
  "scored": false,
  "status": "format ok"
}


Visualize ts

In [15]:
!python -m src.visualization.visualize_val_predictions \
  --checkpoint "$DRIVE_OUTPUTS/checkpoints/resnext50_32x4d_offline_pseudo_highconf/best_multitask.pt"\
  --data-root "$DATA_ROOT" \
  --split val \
  --image-size 448 \
  --num-samples 12 \
  --output-dir "$DRIVE_OUTPUTS/figures/resnext50_32x4d_offline_pseudo_highconf_visual"

/content/CSE-164FinalProject/src/visualization/visualize_val_predictions.py:26: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(colors, mode="RGB")
/content/CSE-164FinalProject/src/visualization/visualize_val_predictions.py:40: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(diff, mode="RGB")
Wrote /content/drive/MyDrive/CSE164FinalProject/outputs/figures/resnext50_32x4d_offline_pseudo_highconf_visual/val_prediction_000_val_00000_class_000.jpg
Wrote /content/drive/MyDrive/CSE164FinalProject/outputs/figures/resnext50_32x4d_offline_pseudo_highconf_visual/val_prediction_001_val_00001_class_000.jpg
Wrote /content/drive/MyDrive/CSE164FinalProject/outputs/figures/resnext50_32x4d_offline_pseudo_highconf_visual/val_prediction_002_val_00002_class_001.jpg
Wrote /content/drive/MyDrive/CSE164FinalProject/outputs/figures/resnext50_32x4d_offline_pseudo_h

Offline Pseudo class only

In [16]:
!python -m src.training.train_multitask_fixmatch \
  --data-root "$DATA_ROOT" \
  --resume-checkpoint "$DRIVE_OUTPUTS/checkpoints/resnext50_32x4d_multitask_384_scratch_e130/best_multitask.pt" \
  --checkpoint-dir "$DRIVE_OUTPUTS/checkpoints/resnext50_32x4d_ssl_cls_only_448_conf088" \
  --image-size 448 \
  --epochs 7 \
  --warmup-epochs 0 \
  --batch-size 16 \
  --unlabeled-batch-size 32 \
  --unlabeled-ratio 2.0 \
  --steps-per-epoch 400 \
  --val-batch-size 4 \
  --num-workers 4 \
  --learning-rate 3e-5 \
  --min-learning-rate 1e-6 \
  --weight-decay 1e-2 \
  --label-smoothing 0.1 \
  --unlabeled-loss-weight 0.25 \
  --confidence-threshold 0.88 \
  --ema-decay 0.9998 \
  --gradient-clip 1.0 \
  --class-only-sample-weight 1.0 \
  --mask-sample-weight 2.5 \
  --teacher-precision fp32 \
  --student-precision bf16 \
  --train-classifier-only \
  --validation-threshold 0.60 \
  --validate-every 1 \
  --full-val-every 1 \
  --print-every 50

Using device: cuda; mixed precision: True
Loading supervised checkpoint: /content/drive/MyDrive/CSE164FinalProject/outputs/checkpoints/resnext50_32x4d_multitask_384_scratch_e130/best_multitask.pt
Supervised samples: 10500; unlabeled samples: 50000; val samples: 750
SSL: threshold=0.88, unlabeled_weight=0.25, DA=False, ssl_seg_forward=True, teacher_precision=fp32, student_precision=bf16, train_classifier_only=True
Trainable parameters: 1,237,292/44,533,165

Epoch 1/7
  ssl step 0050/400 sup_cls=1.6487 sup_seg=-0.0000 unsup=-0.0000 accept=0.007 raw_accept=0.007 bad_teacher=0.000 raw_conf=0.158/0.956 adj_conf=0.158/0.956 accepted_conf=0.921
  ssl step 0100/400 sup_cls=1.0857 sup_seg=-0.0000 unsup=-0.0000 accept=0.009 raw_accept=0.009 bad_teacher=0.000 raw_conf=0.162/0.981 adj_conf=0.162/0.981 accepted_conf=0.916
  ssl step 0150/400 sup_cls=1.4061 sup_seg=-0.0000 unsup=-0.0000 accept=0.009 raw_accept=0.009 bad_teacher=0.000 raw_conf=0.162/0.981 adj_conf=0.162/0.981 accepted_conf=0.919
  ss

Cutmix attempt fix match

In [31]:
import os
from google.colab import drive

drive.mount("/content/drive")

os.environ["DRIVE_OUTPUTS"] = "/content/drive/MyDrive/CSE164FinalProject/outputs"
os.environ["DATA_ROOT"] = "/content/data/raw"

DRIVE_OUTPUTS = os.environ["DRIVE_OUTPUTS"]
DATA_ROOT = os.environ["DATA_ROOT"]

print("DRIVE_OUTPUTS =", DRIVE_OUTPUTS)
print("DATA_ROOT =", DATA_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_OUTPUTS = /content/drive/MyDrive/CSE164FinalProject/outputs
DATA_ROOT = /content/data/raw


Maybe try to train a new classifier instead?

In [28]:
CLS_DIR="$DRIVE_OUTPUTS/checkpoints/convnext_classifier_strong_384_combined_crops"

!python -m src.training.train_classification \
  --data-root "$DATA_ROOT" \
  --split train_combined \
  --image-size 384 \
  --epochs 100 \
  --batch-size 48 \
  --num-workers 8 \
  --learning-rate 4e-4 \
  --min-learning-rate 1e-6 \
  --weight-decay 8e-2 \
  --label-smoothing 0.1 \
  --mixup-alpha 0.2 \
  --cutmix-alpha 1.0 \
  --mix-prob 0.5 \
  --random-erasing-prob 0.25 \
  --ema-decay 0.9998 \
  --grad-clip-norm 1.0 \
  --augment-policy strong \
  --include-seg-crops \
  --crop-padding 0.20 \
  --balanced-class-batches \
  --base-channels 96 \
  --depths "3,3,9,3" \
  --mlp-ratio 4 \
  --drop-path 0.05 \
  --checkpoint-dir "$CLS_DIR"

Using device: cuda; mixed precision: True
Classifier: ConvNeXt-style from scratch base_channels=96, depths=(3, 3, 9, 3), mlp_ratio=4
Train samples: 13500; val samples: 750

Epoch 1/100
  train batch 0020/281 loss=6.2305
  train batch 0040/281 loss=5.9184
  train batch 0060/281 loss=5.8475
  train batch 0080/281 loss=5.7407
  train batch 0100/281 loss=5.8408
  train batch 0120/281 loss=5.7514
  train batch 0140/281 loss=5.7534
  train batch 0160/281 loss=5.8234
  train batch 0180/281 loss=5.7677
  train batch 0200/281 loss=5.8123
  train batch 0220/281 loss=5.6256
  train batch 0240/281 loss=5.6879
  train batch 0260/281 loss=5.7796
  train batch 0280/281 loss=5.7673
  train batch 0281/281 loss=5.7108
  train_loss=5.8250 val_loss=5.6987 acc=0.0093 top5=0.0387 macro_acc=0.0083 conf=0.0178 lr=0.000400
  ema_loss=5.8346 ema_acc=0.0053 ema_top5=0.0200 ema_macro=0.0056 ema_conf=0.0186
  saved new best checkpoint with macro_acc=0.0083
  saved new best EMA checkpoint with macro_acc=0.0056

Epo

Classifer only second chance

In [58]:
CLS2_DIR = f"{DRIVE_OUTPUTS}/checkpoints/convnext_classifier_light_448_combined_crops_seed2"

!python -u -m src.training.train_classification \
  --data-root "$DATA_ROOT" \
  --split train_combined \
  --image-size 448 \
  --epochs 110 \
  --batch-size 48 \
  --num-workers 8 \
  --learning-rate 2e-4 \
  --min-learning-rate 1e-6 \
  --weight-decay 5e-2 \
  --label-smoothing 0.05 \
  --mixup-alpha 0.1 \
  --cutmix-alpha 0.5 \
  --mix-prob 0.25 \
  --random-erasing-prob 0.10 \
  --ema-decay 0.9998 \
  --grad-clip-norm 1.0 \
  --augment-policy strong \
  --include-seg-crops \
  --crop-padding 0.20 \
  --balanced-class-batches \
  --base-channels 96 \
  --depths "3,3,9,3" \
  --mlp-ratio 4 \
  --drop-path 0.02 \
  --checkpoint-dir "$CLS2_DIR"

Using device: cuda; mixed precision: True
Classifier: ConvNeXt-style from scratch base_channels=96, depths=(3, 3, 9, 3), mlp_ratio=4
Train samples: 13500; val samples: 750

Epoch 1/110
  train batch 0020/281 loss=6.0461
  train batch 0040/281 loss=5.8445
  train batch 0060/281 loss=5.7552
  train batch 0080/281 loss=5.6972
  train batch 0100/281 loss=5.6851
  train batch 0120/281 loss=5.6880
  train batch 0140/281 loss=5.6130
  train batch 0160/281 loss=5.5638
  train batch 0180/281 loss=5.6684
  train batch 0200/281 loss=5.8475
  train batch 0220/281 loss=5.3626
  train batch 0240/281 loss=5.5612
  train batch 0260/281 loss=5.5095
  train batch 0280/281 loss=5.6413
  train batch 0281/281 loss=5.5669
  train_loss=5.7029 val_loss=5.5066 acc=0.0093 top5=0.0640 macro_acc=0.0100 conf=0.0280 lr=0.000200
  ema_loss=5.8074 ema_acc=0.0053 ema_top5=0.0227 ema_macro=0.0067 ema_conf=0.0128
  saved new best checkpoint with macro_acc=0.0100
  saved new best EMA checkpoint with macro_acc=0.0067

Epo

EMA fine tuning

In [36]:
BASE_CKPT="$DRIVE_OUTPUTS/checkpoints/convnext_classifier_strong_384_combined_crops/best_ema_classification.pt"
CLS_DIR="$DRIVE_OUTPUTS/checkpoints/convnext_classifier_448_ema_finetune"

!python -m src.training.train_classification \
  --data-root "$DATA_ROOT" \
  --split train_combined \
  --resume-checkpoint "$BASE_CKPT" \
  --image-size 448 \
  --epochs 8 \
  --batch-size 24 \
  --num-workers 8 \
  --learning-rate 3e-5 \
  --min-learning-rate 1e-6 \
  --weight-decay 5e-2 \
  --label-smoothing 0.05 \
  --mixup-alpha 0.05 \
  --cutmix-alpha 0.5 \
  --mix-prob 0.25 \
  --random-erasing-prob 0.10 \
  --ema-decay 0.9998 \
  --grad-clip-norm 1.0 \
  --augment-policy strong \
  --include-seg-crops \
  --crop-padding 0.20 \
  --balanced-class-batches \
  --base-channels 96 \
  --depths "3,3,9,3" \
  --mlp-ratio 4 \
  --drop-path 0.05 \
  --checkpoint-dir "$CLS_DIR"

Using device: cuda; mixed precision: True
Classifier: ConvNeXt-style from scratch base_channels=96, depths=(3, 3, 9, 3), mlp_ratio=4
Train samples: 13500; val samples: 750
Loaded classifier checkpoint: /content/drive/MyDrive/CSE164FinalProject/outputs/checkpoints/convnext_classifier_strong_384_combined_crops/best_ema_classification.pt

Epoch 1/8
  train batch 0020/562 loss=0.6139
  train batch 0040/562 loss=0.5993
  train batch 0060/562 loss=0.7089
  train batch 0080/562 loss=0.6439
  train batch 0100/562 loss=0.7514
  train batch 0120/562 loss=0.7951
  train batch 0140/562 loss=0.7568
  train batch 0160/562 loss=0.5902
  train batch 0180/562 loss=1.6639
  train batch 0200/562 loss=0.9034
  train batch 0220/562 loss=0.8194
  train batch 0240/562 loss=1.0064
  train batch 0260/562 loss=0.5881
  train batch 0280/562 loss=0.7070
  train batch 0300/562 loss=0.5725
  train batch 0320/562 loss=0.6081
  train batch 0340/562 loss=0.6099
  train batch 0360/562 loss=0.6238
  train batch 0380/562

EMA fine tune part 2

In [60]:
BASE_CLS2_CKPT = f"{DRIVE_OUTPUTS}/checkpoints/convnext_classifier_light_448_combined_crops_seed2/best_ema_classification.pt"
CLS2_512_DIR = f"{DRIVE_OUTPUTS}/checkpoints/convnext_classifier_light_512_seed2_finetune"

!python -u -m src.training.train_classification \
  --data-root "$DATA_ROOT" \
  --split train_combined \
  --resume-checkpoint "$BASE_CLS2_CKPT" \
  --image-size 512 \
  --epochs 6 \
  --batch-size 18 \
  --num-workers 8 \
  --learning-rate 8e-6 \
  --min-learning-rate 1e-6 \
  --weight-decay 2e-2 \
  --label-smoothing 0.03 \
  --mixup-alpha 0.10 \
  --cutmix-alpha 0.25 \
  --mix-prob 0.15 \
  --random-erasing-prob 0.07 \
  --ema-decay 0.9998 \
  --grad-clip-norm 1.0 \
  --augment-policy strong \
  --include-seg-crops \
  --crop-padding 0.26 \
  --balanced-class-batches \
  --base-channels 96 \
  --depths "3,3,9,3" \
  --mlp-ratio 4 \
  --drop-path 0.02 \
  --checkpoint-dir "$CLS2_512_DIR"

Using device: cuda; mixed precision: True
Classifier: ConvNeXt-style from scratch base_channels=96, depths=(3, 3, 9, 3), mlp_ratio=4
Train samples: 13500; val samples: 750
Loaded classifier checkpoint: /content/drive/MyDrive/CSE164FinalProject/outputs/checkpoints/convnext_classifier_light_448_combined_crops_seed2/best_ema_classification.pt

Epoch 1/6
  train batch 0020/750 loss=0.3417
  train batch 0040/750 loss=0.6092
  train batch 0060/750 loss=0.3893
  train batch 0080/750 loss=0.3518
  train batch 0100/750 loss=0.3622
  train batch 0120/750 loss=0.3447
  train batch 0140/750 loss=0.3415
  train batch 0160/750 loss=0.3370
  train batch 0180/750 loss=0.3376
  train batch 0200/750 loss=0.3417
  train batch 0220/750 loss=0.4583
  train batch 0240/750 loss=0.3382
  train batch 0260/750 loss=0.3549
  train batch 0280/750 loss=0.3482
  train batch 0300/750 loss=0.3472
  train batch 0320/750 loss=0.3923
  train batch 0340/750 loss=0.3875
  train batch 0360/750 loss=0.4264
  train batch 038

EMa fine tune part 3

In [62]:
BASE_CLS2_CKPT = f"{DRIVE_OUTPUTS}/checkpoints/convnext_classifier_light_448_combined_crops_seed2/best_ema_classification.pt"
CLS2_512_DIR = f"{DRIVE_OUTPUTS}/checkpoints/convnext_classifier_light_512_seed2_finetune_longer"

!python -u -m src.training.train_classification \
  --data-root "$DATA_ROOT" \
  --split train_combined \
  --resume-checkpoint "$BASE_CLS2_CKPT" \
  --image-size 512 \
  --epochs 12 \
  --batch-size 18 \
  --num-workers 8 \
  --learning-rate 1.2e-5 \
  --min-learning-rate 3e-6 \
  --weight-decay 2e-2 \
  --label-smoothing 0.03 \
  --mixup-alpha 0.10 \
  --cutmix-alpha 0.20 \
  --mix-prob 0.13 \
  --random-erasing-prob 0.07 \
  --ema-decay 0.9998 \
  --grad-clip-norm 1.0 \
  --augment-policy strong \
  --include-seg-crops \
  --crop-padding 0.26 \
  --balanced-class-batches \
  --base-channels 96 \
  --depths "3,3,9,3" \
  --mlp-ratio 4 \
  --drop-path 0.02 \
  --checkpoint-dir "$CLS2_512_DIR"

Using device: cuda; mixed precision: True
Classifier: ConvNeXt-style from scratch base_channels=96, depths=(3, 3, 9, 3), mlp_ratio=4
Train samples: 13500; val samples: 750
Loaded classifier checkpoint: /content/drive/MyDrive/CSE164FinalProject/outputs/checkpoints/convnext_classifier_light_448_combined_crops_seed2/best_ema_classification.pt

Epoch 1/12
  train batch 0020/750 loss=0.3898
  train batch 0040/750 loss=0.3578
  train batch 0060/750 loss=0.4316
  train batch 0080/750 loss=0.3557
  train batch 0100/750 loss=0.3428
  train batch 0120/750 loss=0.3400
  train batch 0140/750 loss=0.3403
  train batch 0160/750 loss=0.3615
  train batch 0180/750 loss=0.3518
  train batch 0200/750 loss=0.3459
  train batch 0220/750 loss=0.4360
  train batch 0240/750 loss=1.4320
  train batch 0260/750 loss=0.3429
  train batch 0280/750 loss=0.4432
  train batch 0300/750 loss=0.3551
  train batch 0320/750 loss=0.3771
  train batch 0340/750 loss=0.3459
  train batch 0360/750 loss=0.3798
  train batch 03

Threshold sweeping ensemble seed 2

In [68]:
BASE_CKPT = f"{DRIVE_OUTPUTS}/checkpoints/resnext50_32x4d_multitask_384_scratch_e130/best_multitask.pt"
CLS2_CKPT = f"{DRIVE_OUTPUTS}/checkpoints/convnext_classifier_448_ema_finetune/best_ema_classification.pt"
blend_weights = [0.28, 0.30, 0.32, 0.33, 0.34, 0.35, 0.37]
thresholds = "0.55,0.5625,0.575,0.5875,0.60"

for w in blend_weights:
    print("\n" + "=" * 80)
    print(f"Classifier blend weight = {w:.2f}")
    print("=" * 80)
    !python -m src.training.tune_multitask_threshold \
      --seg-checkpoint "$BASE_CKPT" \
      --classifier-checkpoint "$CLS2_CKPT" \
      --classifier-blend-weight {w} \
      --data-root "$DATA_ROOT" \
      --image-size 384 \
      --batch-size 32 \
      --num-workers 12 \
      --thresholds "$thresholds" \
      --tta hflip


Classifier blend weight = 0.28
  scored batch 0020/24
  scored batch 0024/24
threshold=0.550 automated=0.2326 seg=0.2200 mIoU=0.2061 boundary=0.3079 rare=0.1414 macro_acc=0.3928
threshold=0.562 automated=0.2326 seg=0.2200 mIoU=0.2061 boundary=0.3078 rare=0.1416 macro_acc=0.3928
threshold=0.575 automated=0.2326 seg=0.2200 mIoU=0.2062 boundary=0.3076 rare=0.1418 macro_acc=0.3928
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/CSE-164FinalProject/src/training/tune_multitask_threshold.py", line 165, in <module>
    main()
  File "/content/CSE-164FinalProject/src/training/tune_multitask_threshold.py", line 150, in main
    score_thresholds(
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/content/CSE-164FinalProject/src/training/tune_multitask_threshol

ensemble submission

In [69]:
BASE_CKPT = f"{DRIVE_OUTPUTS}/checkpoints/resnext50_32x4d_multitask_384_scratch_e130/best_multitask.pt"
CLS2_CKPT = f"{DRIVE_OUTPUTS}/checkpoints/convnext_classifier_448_ema_finetune/best_ema_classification.pt"
!python -m src.training.predict_multitask \
  --checkpoint "$BASE_CKPT" \
  --classifier-checkpoint "$CLS_CKPT" \
  --classifier-blend-weight 0.32 \
  --data-root "$DATA_ROOT" \
  --output "$DRIVE_OUTPUTS/submissions/resnext116_convnextema_blend032_thr05625_hflip.csv" \
  --image-size 384 \
  --batch-size 6 \
  --num-workers 8 \
  --seg-threshold 0.5625 \
  --tta hflip

  predicted batch 0020/500
  predicted batch 0040/500
  predicted batch 0060/500
  predicted batch 0080/500
  predicted batch 0100/500
  predicted batch 0120/500
  predicted batch 0140/500
  predicted batch 0160/500
  predicted batch 0180/500
  predicted batch 0200/500
  predicted batch 0220/500
  predicted batch 0240/500
  predicted batch 0260/500
  predicted batch 0280/500
  predicted batch 0300/500
  predicted batch 0320/500
  predicted batch 0340/500
  predicted batch 0360/500
  predicted batch 0380/500
  predicted batch 0400/500
  predicted batch 0420/500
  predicted batch 0440/500
  predicted batch 0460/500
  predicted batch 0480/500
  predicted batch 0500/500
Wrote /content/drive/MyDrive/CSE164FinalProject/outputs/submissions/resnext116_convnextema_blend032_thr05625_hflip.csv
{
  "rows": 3000,
  "scored": false,
  "status": "format ok"
}


ensemble 1.1

In [75]:
!python -m src.training.predict_multitask \
  --checkpoint "$BASE_CKPT" \
  --classifier-checkpoint "$CLS2_CKPT" \
  --classifier-blend-weight 0.33 \
  --data-root "$DATA_ROOT" \
  --output "$DRIVE_OUTPUTS/submissions/resnext116_cls448_blend033_thr0575_multicrop.csv" \
  --image-size 384 \
  --batch-size 4 \
  --num-workers 8 \
  --seg-threshold 0.575 \
  --tta multi_crop

  predicted batch 0020/750
  predicted batch 0040/750
  predicted batch 0060/750
  predicted batch 0080/750
  predicted batch 0100/750
  predicted batch 0120/750
  predicted batch 0140/750
  predicted batch 0160/750
  predicted batch 0180/750
  predicted batch 0200/750
  predicted batch 0220/750
  predicted batch 0240/750
  predicted batch 0260/750
  predicted batch 0280/750
  predicted batch 0300/750
  predicted batch 0320/750
  predicted batch 0340/750
  predicted batch 0360/750
  predicted batch 0380/750
  predicted batch 0400/750
  predicted batch 0420/750
  predicted batch 0440/750
  predicted batch 0460/750
  predicted batch 0480/750
  predicted batch 0500/750
  predicted batch 0520/750
  predicted batch 0540/750
  predicted batch 0560/750
  predicted batch 0580/750
  predicted batch 0600/750
  predicted batch 0620/750
  predicted batch 0640/750
  predicted batch 0660/750
  predicted batch 0680/750
  predicted batch 0700/750
  predicted batch 0720/750
  predicted batch 0740/750
 

ensemble 1.2

In [73]:
!python -m src.training.predict_multitask \
  --checkpoint "$BASE_CKPT" \
  --classifier-checkpoint "$CLS2_CKPT" \
  --classifier-blend-weight 0.32 \
  --data-root "$DATA_ROOT" \
  --output "$DRIVE_OUTPUTS/submissions/resnext116_cls448_blend032_thr05625_hflip_crop040.csv" \
  --image-size 384 \
  --batch-size 6 \
  --num-workers 8 \
  --seg-threshold 0.5625 \
  --tta hflip \
  --classifier-crop-mode seg \
  --classifier-crop-padding 0.20 \
  --classifier-crop-weight 0.40

  predicted batch 0020/500
  predicted batch 0040/500
  predicted batch 0060/500
  predicted batch 0080/500
  predicted batch 0100/500
  predicted batch 0120/500
  predicted batch 0140/500
  predicted batch 0160/500
  predicted batch 0180/500
  predicted batch 0200/500
  predicted batch 0220/500
  predicted batch 0240/500
  predicted batch 0260/500
  predicted batch 0280/500
  predicted batch 0300/500
  predicted batch 0320/500
  predicted batch 0340/500
  predicted batch 0360/500
  predicted batch 0380/500
  predicted batch 0400/500
  predicted batch 0420/500
  predicted batch 0440/500
  predicted batch 0460/500
  predicted batch 0480/500
  predicted batch 0500/500
Wrote /content/drive/MyDrive/CSE164FinalProject/outputs/submissions/resnext116_cls448_blend032_thr05625_hflip_crop040.csv
{
  "rows": 3000,
  "scored": false,
  "status": "format ok"
}


ensemble 1.3

In [74]:
!python -m src.training.predict_multitask \
  --checkpoint "$BASE_CKPT" \
  --classifier-checkpoint "$CLS2_CKPT" \
  --classifier-blend-weight 0.33 \
  --data-root "$DATA_ROOT" \
  --output "$DRIVE_OUTPUTS/submissions/resnext116_cls448_blend033_thr0575_hflip_crop060.csv" \
  --image-size 384 \
  --batch-size 6 \
  --num-workers 8 \
  --seg-threshold 0.575 \
  --tta hflip \
  --classifier-crop-mode seg \
  --classifier-crop-padding 0.20 \
  --classifier-crop-weight 0.60

  predicted batch 0020/500
  predicted batch 0040/500
  predicted batch 0060/500
  predicted batch 0080/500
  predicted batch 0100/500
  predicted batch 0120/500
  predicted batch 0140/500
  predicted batch 0160/500
  predicted batch 0180/500
  predicted batch 0200/500
  predicted batch 0220/500
  predicted batch 0240/500
  predicted batch 0260/500
  predicted batch 0280/500
  predicted batch 0300/500
  predicted batch 0320/500
  predicted batch 0340/500
  predicted batch 0360/500
  predicted batch 0380/500
  predicted batch 0400/500
  predicted batch 0420/500
  predicted batch 0440/500
  predicted batch 0460/500
  predicted batch 0480/500
  predicted batch 0500/500
Wrote /content/drive/MyDrive/CSE164FinalProject/outputs/submissions/resnext116_cls448_blend033_thr0575_hflip_crop060.csv
{
  "rows": 3000,
  "scored": false,
  "status": "format ok"
}


Ensemble 1.4

Visualize ts ensemmble

In [71]:
!python -m src.visualization.visualize_val_predictions \
  --checkpoint "$BASE_CKPT" \
  --classifier-checkpoint "$CLS_CKPT" \
  --classifier-blend-weight 0.32 \
  --data-root "$DATA_ROOT" \
  --split val \
  --image-size 384 \
  --num-samples 20 \
  --output-dir "$DRIVE_OUTPUTS/figures/final_sub_blend032_val05626_images" \
  --seg-threshold 0.5625 \
  --tta hflip

/content/CSE-164FinalProject/src/visualization/visualize_val_predictions.py:30: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(colors, mode="RGB")
/content/CSE-164FinalProject/src/visualization/visualize_val_predictions.py:44: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(diff, mode="RGB")
Wrote /content/drive/MyDrive/CSE164FinalProject/outputs/figures/final_sub_blend032_val05626_images/val_prediction_000_val_00000_class_000.jpg
Wrote /content/drive/MyDrive/CSE164FinalProject/outputs/figures/final_sub_blend032_val05626_images/val_prediction_001_val_00001_class_000.jpg
Wrote /content/drive/MyDrive/CSE164FinalProject/outputs/figures/final_sub_blend032_val05626_images/val_prediction_002_val_00002_class_001.jpg
Wrote /content/drive/MyDrive/CSE164FinalProject/outputs/figures/final_sub_blend032_val05626_images/val_prediction_003_val_00003_clas

One more ssl attempt

In [63]:
SSL_DIR = f"{DRIVE_OUTPUTS}/checkpoints/resnext50_online_ssl_ensemble_teacher_blend033_cls_only_thr060"

!python -m src.training.train_multitask_fixmatch \
  --data-root "$DATA_ROOT" \
  --resume-checkpoint "$BASE_CKPT" \
  --checkpoint-dir "$SSL_DIR" \
  --image-size 384 \
  --epochs 6 \
  --warmup-epochs 1 \
  --batch-size 24 \
  --unlabeled-batch-size 48 \
  --unlabeled-ratio 2.0 \
  --steps-per-epoch 400 \
  --val-batch-size 6 \
  --num-workers 8 \
  --learning-rate 7e-6 \
  --min-learning-rate 5e-7 \
  --weight-decay 1e-2 \
  --label-smoothing 0.1 \
  --unlabeled-loss-weight 0.03 \
  --confidence-threshold 0.60 \
  --ema-decay 0.9998 \
  --gradient-clip 1.0 \
  --class-only-sample-weight 1.0 \
  --mask-sample-weight 2.5 \
  --teacher-classifier-checkpoint "$CLS_CKPT" \
  --teacher-classifier-blend-weight 0.33 \
  --teacher-classifier-tta none \
  --teacher-precision fp32 \
  --student-precision bf16 \
  --train-classifier-only \
  --validation-threshold 0.575 \
  --validate-every 1 \
  --full-val-every 1 \
  --print-every 50

Using device: cuda; mixed precision: True
Loading supervised checkpoint: /content/drive/MyDrive/CSE164FinalProject/outputs/checkpoints/resnext50_32x4d_multitask_384_scratch_e130/best_multitask.pt
Supervised samples: 10500; unlabeled samples: 50000; val samples: 750
SSL: threshold=0.6, unlabeled_weight=0.25, DA=True, ssl_seg_forward=True, teacher_precision=fp32, student_precision=bf16, teacher_classifier_count=0, teacher_classifier_blend_weight=0.0, teacher_classifier_tta=none, train_classifier_only=False, classifier_mixup_alpha=0.2, classifier_cutmix_alpha=1.0, classifier_mix_prob=0.35, classifier_foreground_cutmix=True
Trainable parameters: 44,533,165/44,533,165

Epoch 1/12
  ssl step 0050/400 sup_cls=0.9749 sup_seg=0.1408 unsup=4.6250 accept=0.044 raw_accept=0.044 bad_teacher=0.000 raw_conf=0.152/0.986 adj_conf=0.152/0.985 accepted_conf=0.790 classifier_mix=none
  ssl step 0100/400 sup_cls=0.9777 sup_seg=0.1964 unsup=1.7979 accept=0.052 raw_accept=0.052 bad_teacher=0.000 raw_conf=0.1

REsnext attempt basline

In [50]:
!python -m src.training.train_multitask \
  --data-root "$DATA_ROOT" \
  --architecture resnext50_32x4d \
  --stage joint \
  --epochs 120 \
  --warmup-epochs 3 \
  --image-size 384 \
  --num-segmentation-classes 1 \
  --seg-batch-size 12 \
  --cls-batch-size 64 \
  --val-batch-size 32 \
  --num-workers 12 \
  --learning-rate 1e-3 \
  --min-learning-rate 1e-6 \
  --weight-decay 5e-2 \
  --label-smoothing 0.1 \
  --segmentation-loss-weight 1.0 \
  --dice-loss-weight 1.0 \
  --seg-classification-loss-weight 1.0 \
  --cls-loss-weight 1.0 \
  --gradient-clip 2.0 \
  --ema-decay 0.9998 \
  --weighted-combined-sampling \
  --class-only-sample-weight 1.0 \
  --mask-sample-weight 2.5 \
  --validation-threshold 0.50 \
  --checkpoint-dir "$DRIVE_OUTPUTS/checkpoints/resnext50_32x4d_multitask_384_scratch_e130"

Using device: cuda; mixed precision: True
Multi-task ResNeXt-50 32x4d from scratch: image_size=384, decoder=unet, num_segmentation_classes=1, classifier_dropout=0.2
Train samples: segmentation=3000, classification=10500; val=750
LR scheduler: linear warmup for 3 epochs to 0.001, then cosine decay to 1e-06
Training stage: joint

Epoch 1/120
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/CSE-164FinalProject/src/training/train_multitask.py", line 920, in <module>
    main()
  File "/content/CSE-164FinalProject/src/training/train_multitask.py", line 897, in main
    run_one_stage(args)
  File "/content/CSE-164FinalProject/src/training/train_multitask.py", line 736, in run_one_stage
    mixed_metrics = train_joint_mixed_batches(
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/CSE-164FinalProject/src/training/train_multitask.py", line 305, in train_joint_mixed_batches